# M2 — Anti-Curriculum (FIXED)

**Fixes vs original M2:**
1. **LR warmup at each phase transition** — prevents loss explosion when switching from sparse (decision-only) to dense (full-token) supervision
2. **Phase 2 ramps new loss from 0→1 over 10% of its duration** — smooth transition instead of instant switch
3. **Trains ONE model at a time** — avoids VRAM fragmentation that caused 3B to run on CPU
4. **Pick which model to train** in Cell 1 — restart kernel between models

Phase 1 (40%): `L_dec_only` — build latent reasoning
Phase 2 (40%): smooth blend → `L_cw_wsft`
Phase 3 (20%): `L_cw_wsft + L_dec_ent`

In [1]:
# Cell 1: Config — PICK ONE MODEL, restart kernel between models
# Uncomment the one you want to train:

#TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"
TRAIN_MODEL = "qwen25_7b_base"

print(f"Will train: {TRAIN_MODEL}")
print("⚠️ Restart kernel before switching to a different model!")

Will train: qwen25_7b_base
⚠️ Restart kernel before switching to a different model!


In [2]:
# Cell 2: Imports and setup
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer, TrainerCallback

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, EPOCHS, LR, BATCH_SIZE, GRAD_ACC, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA, STUDENTS,
    load_data, load_student,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m2_anti_curriculum")
os.makedirs(OUT_DIR, exist_ok=True)

PHASE1_END = 0.40
PHASE2_END = 1.0
PHASE_RAMP = 0.10  # 10% warmup ramp within each phase transition

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)

assert TRAIN_MODEL in STUDENTS, f"{TRAIN_MODEL} not in STUDENTS"
print(f"Phases: dec-only [0, {PHASE1_END}], blend [{PHASE1_END}, {PHASE2_END}], full [{PHASE2_END}, 1.0]")
print(f"Phase ramp: {PHASE_RAMP*100:.0f}% warmup at each transition")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Phases: dec-only [0, 0.4], blend [0.4, 1.0], full [1.0, 1.0]
Phase ramp: 10% warmup at each transition


In [3]:
""" # Cell 3: Dataset
class M2Dataset(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        self.rows, self.prompts, self.tokenizer = rows, prompts, tokenizer
        self.is_instruct, self.mean_c = is_instruct, mean_c

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = self.prompts[r["id"]]
        answer = r["teacher_text"]
        prompt_text = build_prompt_text(self.tokenizer, prompt, self.is_instruct)
        input_ids, offsets, labels, answer_start = tokenize_and_mask(self.tokenizer, prompt_text, answer)

        # Full-answer labels (for phases 2+3)
        # Also build decision-only labels (for phase 1)
        wsft_weights = [0.0] * len(input_ids)
        spans = get_section_spans(answer)
        dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
        expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
        for i, (s, e) in enumerate(offsets):
            if e <= s: continue
            if s >= answer_start:
                w = W_FORMAT
                if in_any_span(s, e, dec_spans): w = W_DECISION
                if in_any_span(s, e, expl_spans): w = W_EXPL
                wsft_weights[i] = float(w)
        active_w = [w for w in wsft_weights if w > 0]
        if active_w:
            mean_w = sum(active_w) / len(active_w)
            if mean_w > 1e-6: wsft_weights = [w / mean_w if w > 0 else 0.0 for w in wsft_weights]

        alpha = compute_alpha(r, self.mean_c)
        dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
        dec_span_chars = find_decision_span_chars(answer)
        if dec_span_chars:
            ds, de = dec_span_chars
            full_ds, full_de = answer_start + ds, answer_start + de
            for i, (s, e) in enumerate(offsets):
                if labels[i] != -100 and not (e <= full_ds or s >= full_de):
                    dec_mask[i] = True
        ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
            "alpha": torch.tensor(alpha, dtype=torch.float),
            "dec_mask": dec_mask,
            "ent_teacher": ent_teacher.float(),
        }

print("✅ Dataset ready") """

' # Cell 3: Dataset\nclass M2Dataset(Dataset):\n    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):\n        self.rows, self.prompts, self.tokenizer = rows, prompts, tokenizer\n        self.is_instruct, self.mean_c = is_instruct, mean_c\n\n    def __len__(self): return len(self.rows)\n\n    def __getitem__(self, idx):\n        r = self.rows[idx]\n        prompt = self.prompts[r["id"]]\n        answer = r["teacher_text"]\n        prompt_text = build_prompt_text(self.tokenizer, prompt, self.is_instruct)\n        input_ids, offsets, labels, answer_start = tokenize_and_mask(self.tokenizer, prompt_text, answer)\n\n        # Full-answer labels (for phases 2+3)\n        # Also build decision-only labels (for phase 1)\n        wsft_weights = [0.0] * len(input_ids)\n        spans = get_section_spans(answer)\n        dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]\n        expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["ex

In [4]:
# Cell 3b: FAST dataset — precomputes all items once
from tqdm import tqdm

class M2DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset (one-time cost)...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(
                tokenizer, prompt_text, answer)

            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
            for i, (s, e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s, e, dec_spans): w = W_DECISION
                    if in_any_span(s, e, expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mean_w = sum(active_w) / len(active_w)
                if mean_w > 1e-6:
                    wsft_weights = [w / mean_w if w > 0 else 0.0 for w in wsft_weights]

            alpha = compute_alpha(r, mean_c)
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                full_ds, full_de = answer_start + ds, answer_start + de
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= full_ds or s >= full_de):
                        dec_mask[i] = True
            ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "alpha": torch.tensor(alpha, dtype=torch.float),
                "dec_mask": dec_mask,
                "ent_teacher": ent_teacher.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)

    def __getitem__(self, idx): return self.items[idx]

print("✅ Fast dataset class ready")

✅ Fast dataset class ready


In [5]:
# Cell 4: Trainer — optimized

class PhaseLogger(TrainerCallback):
    def __init__(self): self.last_phase = None
    def on_step_begin(self, args, state, control, **kwargs):
        if state.max_steps <= 0: return
        p = state.global_step / state.max_steps
        phase = "Phase1(dec-only)" if p < PHASE1_END else "Phase2(blend)" if p < PHASE2_END else "Phase3(full)"
        if phase != self.last_phase:
            print(f"\n🔄 Step {state.global_step}/{state.max_steps} ({p:.0%}): {phase}")
            self.last_phase = phase

class M2Trainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_loss = torch.nn.CrossEntropyLoss(reduction="none")  # ← fix #2: cache

    def _entropy_on_mask(self, logits, mask):
        """Memory-efficient: only softmax masked positions (keep the loop)."""
        B = logits.size(0)
        ent = torch.zeros(B, device=logits.device)
        for b in range(B):
            idx = mask[b].nonzero(as_tuple=True)[0]
            if len(idx) == 0:
                continue
            local_logits = logits[b, idx, :]
            local_probs = torch.softmax(local_logits, dim=-1)
            local_ent = -(local_probs * torch.log(local_probs + 1e-9)).sum(-1)
            ent[b] = local_ent.mean()
        return ent

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_weights = inputs.pop("wsft_weights")
        alpha = inputs.pop("alpha")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")

        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_wsft_w = wsft_weights[:, 1:].contiguous()
        shift_dec_mask = dec_mask[:, 1:]
        vocab, B = shift_logits.size(-1), shift_logits.size(0)

        token_loss = self._ce_loss(                    # ← fix #2: reuse
            shift_logits.view(-1, vocab), shift_labels.view(-1)).view(shift_labels.size())
        active = (shift_labels != -100).float()

        progress = self.state.global_step / max(1, self.state.max_steps)

        def get_L_dec():
            dec_active = shift_dec_mask.float() * active
            return (token_loss * dec_active).sum() / dec_active.sum().clamp(min=1.0)

        def get_L_cw():
            w = shift_wsft_w * active
            denom = active.sum(dim=1).clamp(min=1.0)
            per_sample = (token_loss * w).sum(dim=1) / denom
            return (per_sample * alpha).mean()

        def get_L_ent():
            dec_active_mask = shift_dec_mask & (shift_labels != -100)
            ent_student = self._entropy_on_mask(shift_logits, dec_active_mask)
            return LAMBDA * ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()

        if progress < PHASE1_END:
            loss = get_L_dec()
        elif progress < PHASE2_END:
            phase2_progress = (progress - PHASE1_END) / (PHASE2_END - PHASE1_END)
            ramp_frac = PHASE_RAMP / (PHASE2_END - PHASE1_END)
            if phase2_progress < ramp_frac:
                blend_t = phase2_progress / ramp_frac * 0.3
            else:
                blend_t = 0.3 + 0.7 * (phase2_progress - ramp_frac) / (1.0 - ramp_frac)
            L_dec = get_L_dec()
            L_cw = get_L_cw()
            with torch.no_grad():
                scale = (L_dec.detach() / L_cw.detach().clamp(min=1e-6)).clamp(0.1, 10.0)
            loss = (1.0 - blend_t) * L_dec + blend_t * (scale * L_cw)
        else:
            phase3_progress = (progress - PHASE2_END) / (1.0 - PHASE2_END)
            ramp_frac = PHASE_RAMP / (1.0 - PHASE2_END)
            ent_weight = min(1.0, phase3_progress / max(ramp_frac, 1e-6))
            loss = get_L_cw() + ent_weight * get_L_ent()

        return (loss, outputs) if return_outputs else loss

print("✅ Trainer ready")

✅ Trainer ready


In [6]:
# Cell 5: Train the selected model
cfg = STUDENTS[TRAIN_MODEL]
print(f"\n{'='*50} M2: {TRAIN_MODEL} {'='*50}")

out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

tokenizer, model = load_student(TRAIN_MODEL, cfg)
#model = torch.compile(model)  # ← fix #4: fuse CUDA kernels, ~20-30% speedup
dataset = M2DatasetFast(teacher_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights", "dec_mask"],
                            extra_scalar_fields=["alpha", "ent_teacher"])

# In the training cell, right before TrainingArguments:
if "1p5b" in TRAIN_MODEL:
    bs, ga = 4, 8
elif "3b" in TRAIN_MODEL:
    bs, ga = 2, 16
else:  # 7b
    bs, ga = 1, 32

trainer = M2Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=out_path, num_train_epochs=EPOCHS,
        per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
        learning_rate=LR, logging_steps=25, save_strategy="no",
        bf16=True, seed=SEED, report_to="none",
        remove_unused_columns=False, dataloader_num_workers=0,
        warmup_ratio=0.05,  # small warmup at start
    ),
    train_dataset=dataset, data_collator=collator, callbacks=[PhaseLogger()],
)
trainer.train()
model.save_pretrained(out_path)
tokenizer.save_pretrained(out_path)
print(f"\n✅ {TRAIN_MODEL} saved → {out_path}")
print("\n⚠️ To train next model: change TRAIN_MODEL in Cell 1, RESTART KERNEL, run all cells")


================================================== M2: qwen25_7b_base ==================================================
  Loading qwen25_7b_base from Qwen/Qwen2.5-7B...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:10<00:00, 31.47it/s]


  ✅ qwen25_7b_base: 10,092,544 trainable / 7,625,709,056 total params
  Precomputing dataset (one-time cost)...


  Tokenizing: 100%|██████████| 5000/5000 [00:07<00:00, 695.98it/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  ✅ Precomputed 5000 samples

🔄 Step 0/314 (0%): Phase1(dec-only)


Step,Training Loss
25,5.370153
50,2.977961
75,2.748084
100,2.699406
125,2.248930
150,2.564775
175,3.117589
200,3.118607
225,3.997265
250,4.334802



🔄 Step 126/314 (40%): Phase2(blend)

✅ qwen25_7b_base saved → runs\m2_anti_curriculum\qwen25_7b_base

⚠️ To train next model: change TRAIN_MODEL in Cell 1, RESTART KERNEL, run all cells


# M2 — Anti-Curriculum (Two-Phase Version)

## Method
- **Phase 1** (0–40%): `L_dec_only` — decision-token-only supervision builds latent reasoning (E7 finding)
- **Phase 2** (40–100%): smooth blend from `L_dec_only` → `L_cw_wsft` — layers full token supervision without destroying reasoning

## Changes from original three-phase design
- **Removed Phase 3 (entropy matching)** — the entropy loss magnitude was ~10× larger than L_cw at the transition point, causing loss to explode from ~1.3 to ~10.6. Loss normalization alone couldn't stabilize it because the entropy penalty variance is too high on a per-batch basis.
- **Extended Phase 2 to fill 60% of training** (was 40%) — gives more time for the blend from decision-only to full CW-WSFT to stabilize.
- Safety/calibration via entropy matching is covered by M1, M5, and M7 instead.

## Key implementation details
- Phase 2 uses a **smooth ramp** (10% warmup) to avoid gradient magnitude shock when switching from sparse (~5 token) to dense (~500 token) supervision
- Phase 2 **normalizes L_cw to L_dec scale** via a detached ratio to prevent loss explosion at the transition
- **Memory-efficient entropy** (`_entropy_on_mask`) computes softmax only on masked positions — kept in trainer for potential future use but not called in two-phase mode

## Training config
- bf16, LoRA r=16 on q/k/v/o_proj, 2 epochs, LR=2e-4
- One model per kernel session (restart between models to avoid VRAM fragmentation)
- `PHASE1_END = 0.40`, `PHASE2_END = 1.0`